# 06 — Fairness Analysis

Evaluate model performance and fairness metrics across demographic subgroups.

**Requirements:** FR-19 through FR-22 (PRD)

In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.fairness import run_fairness_analysis
from src.feature_engineering import engineer_features, get_feature_columns
from src.modeling import train_test_split_data
from src.utils import data_path, results_path, set_seed

set_seed()
FIG_DIR = results_path('figures')
TABLE_DIR = results_path('tables')
MODEL_DIR = results_path('models')

In [ ]:
# Load data and model
df = pd.read_csv(data_path('processed', 'oasis_merged_final.csv'))
if 'BrainAtrophyRatio' not in df.columns:
    df = engineer_features(df)

feature_cols = [c for c in get_feature_columns() if c in df.columns]
X = df[feature_cols]
y = df['target']
_, X_test, _, y_test = train_test_split_data(X, y)

test_df = df.loc[X_test.index].copy()
pipeline = joblib.load(MODEL_DIR / 'best_model.pkl')
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# FR-19 to FR-22: Subgroup evaluation + fairness metrics
fairness_results = run_fairness_analysis(test_df, y_pred, y_proba, output_dir=FIG_DIR)
fairness_results['fairness_metrics']

In [ ]:
# Display per-subgroup metrics
for group_name, group_df in fairness_results['subgroup_results'].items():
    print(f'\n=== {group_name} ===')
    display(group_df)

In [ ]:
# Save fairness results tables
for group_name, group_df in fairness_results['subgroup_results'].items():
    group_df.to_csv(TABLE_DIR / f'fairness_{group_name}.csv', index=False)

pd.DataFrame([fairness_results['fairness_metrics']]).to_csv(
    TABLE_DIR / 'fairness_global_metrics.csv', index=False
)
print('Fairness tables saved to results/tables/')

## Clinical Implications (FR-22)

Discuss any performance disparities found across gender, age group, and SES subgroups.
Consider mitigation strategies: reweighting, threshold adjustment, or subgroup-specific models.